# Bank Customer Churn Analysis

## Data Loading

In this step, we load the dataset and display the first rows to understand its structure.

In [7]:
from scipy.stats import chi2_contingency
from utils_chart import (
    create_bar_chart,
    create_pie_chart,
    create_histogram,
    create_box_plot,
    create_heatmap
)
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)

In [8]:
import sys
!{sys.executable} -m pip install plotly

In [9]:
import pandas as pd

# Load dataset
df = pd.read_csv("../data/Bank Customer Churn Prediction.csv")

# Display first rows
df.head()

,customer_id,credit_score,country,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn
0,15634602,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,15647311,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,15619304,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,15701354,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,15737888,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


## Data Understanding

In this step, we explore the dataset to understand its structure, data types, and key variables.

In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customer_id       10000 non-null  int64  
 1   credit_score      10000 non-null  int64  
 2   country           10000 non-null  str    
 3   gender            10000 non-null  str    
 4   age               10000 non-null  int64  
 5   tenure            10000 non-null  int64  
 6   balance           10000 non-null  float64
 7   products_number   10000 non-null  int64  
 8   credit_card       10000 non-null  int64  
 9   active_member     10000 non-null  int64  
 10  estimated_salary  10000 non-null  float64
 11  churn             10000 non-null  int64  
dtypes: float64(2), int64(8), str(2)
memory usage: 937.6 KB


In [11]:
df.shape

(10000, 12)

In [12]:
df.describe()

,customer_id,credit_score,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn
count,1.000000e+04,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.00000,10000.000000,10000.000000,10000.000000
mean,1.569094e+07,650.528800,38.921800,5.012800,76485.889288,1.530200,0.70550,0.515100,100090.239881,0.203700
std,7.193619e+04,96.653299,10.487806,2.892174,62397.405202,0.581654,0.45584,0.499797,57510.492818,0.402769
min,1.556570e+07,350.000000,18.000000,0.000000,0.000000,1.000000,0.00000,0.000000,11.580000,0.000000
25%,1.562853e+07,584.000000,32.000000,3.000000,0.000000,1.000000,0.00000,0.000000,51002.110000,0.000000
50%,1.569074e+07,652.000000,37.000000,5.000000,97198.540000,1.000000,1.00000,1.000000,100193.915000,0.000000
75%,1.575323e+07,718.000000,44.000000,7.000000,127644.240000,2.000000,1.00000,1.000000,149388.247500,0.000000
max,1.581569e+07,850.000000,92.000000,10.000000,250898.090000,4.000000,1.00000,1.000000,199992.480000,1.000000


In [13]:
df.isnull().sum()

customer_id         0
credit_score        0
country             0
gender              0
age                 0
tenure              0
balance             0
products_number     0
credit_card         0
active_member       0
estimated_salary    0
churn               0
dtype: int64

## Missing Values Check

The dataset contains no missing values, as confirmed by the null value check.

All columns have complete data, which simplifies the analysis process.

## Data Cleaning

In this step, we clean and prepare the dataset for analysis by removing unnecessary columns and ensuring proper data types.

In [14]:
df.columns

Index(['customer_id', 'credit_score', 'country', 'gender', 'age', 'tenure',
       'balance', 'products_number', 'credit_card', 'active_member',
       'estimated_salary', 'churn'],
      dtype='str')

In [15]:
df.drop(columns=["customer_id"], inplace=True)

In [16]:
df

,credit_score,country,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


## Overall Churn Rate

In this section, we calculate the overall churn rate to understand how many customers are leaving the bank.

In [17]:
churn_distribution = df['churn'].value_counts().reset_index()

churn_distribution.columns = ['churn', 'count']

churn_distribution['churn'] = churn_distribution['churn'].map({
    0: 'Stayed',
    1: 'Churned'
})
fig = create_pie_chart(data=churn_distribution,names_col='churn',values_col='count',title='Customer Churn Distribution')

fig.show()

## Insights

- Approximately 20% of customers have churned
- Most customers remain loyal to the bank
- However, the churn rate is still significant and may impact long-term profitability

## Churn Rate by Customer Activity

In this analysis, we compare churn rates between active and inactive customers.

In [18]:
churn_by_activity = (df.groupby('active_member')['churn'].mean().reset_index())

churn_by_activity['active_member'] = churn_by_activity['active_member'].map({
    0: 'Inactive',
    1: 'Active'
})

# Convert to percentage
churn_by_activity['churn'] = churn_by_activity['churn'] * 100

fig = create_bar_chart(data=churn_by_activity,x_col='active_member',y_col='churn',title='Churn Rate by Activity Status')
fig.update_yaxes(ticksuffix="%")
fig.show()

## Insights

- Inactive customers have a significantly higher churn rate
- Active engagement appears to improve customer retention

## Business Recommendation

- Increase engagement campaigns for inactive customers
- Promote regular usage of banking services

## Churn Analysis by Age

In this section, we analyze the relationship between customer age and churn behavior.

In [19]:
fig = create_box_plot(
    data=df,
    x_col='churn',
    y_col='age',
    title='Age Distribution by Churn Status',
    color_col='churn'
)

fig.show()

## Insights

- Customers who churn tend to be older
- Age appears to be an important factor influencing customer retention

## Business Recommendation

- Develop retention strategies targeting older customers
- Offer personalized banking services for senior clients

## Churn Rate by Number of Products

In this section, we analyze whether the number of bank products used by customers affects churn.

In [20]:
churn_by_products = (df.groupby('products_number')['churn'].mean().reset_index())

churn_by_products['churn'] = churn_by_products['churn'] * 100
fig = create_bar_chart(data=churn_by_products,x_col='products_number',y_col='churn',title='Churn Rate by Number of Products',color_col='products_number')

fig.show()

In [21]:
df['products_number'].value_counts()

products_number
1    5084
2    4590
3     266
4      60
Name: count, dtype: int64

## Insights

- Customers with 2 products have the lowest churn rate
- Customers with 1 product churn significantly more
- Customers with 3 or 4 products show very high churn rates

## Important Observation

- The groups with 3 and 4 products contain relatively few customers
- Therefore, these extreme churn rates should be interpreted carefully

## Business Recommendation

- Encourage customers to adopt a second banking product
- Further investigate high-product customers to better understand dissatisfaction patterns

## Churn Rate by Country

In this section, we analyze churn rates across different countries to identify geographical patterns in customer retention.

In [22]:
churn_by_country = (df.groupby('country')['churn'].mean().reset_index())

# Convert to percentage
churn_by_country['churn'] = churn_by_country['churn'] * 100

churn_by_country
fig = create_bar_chart(data=churn_by_country,x_col='country',y_col='churn',title='Churn Rate by Country',color_col='country')

fig.show()

## Insights

- Germany has the highest churn rate at approximately 32%
- France and Spain show significantly lower churn rates, around 16–17%
- German customers appear much more likely to leave the bank

## Business Interpretation

- The high churn rate in Germany may indicate customer dissatisfaction, stronger competition, or regional behavioral differences
- France and Spain show more stable customer retention patterns

## Business Recommendation

- Investigate the factors driving churn in Germany
- Improve customer engagement and retention strategies for German customers
- Conduct customer satisfaction analysis by region

## Correlation Analysis

In this section, we analyze correlations between numerical variables to identify factors potentially related to customer churn.

In [23]:
numerical_df = df.select_dtypes(include=['int64', 'float64'])
correlation_matrix = numerical_df.corr()

correlation_matrix

,credit_score,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn
credit_score,1.000000,-0.003965,0.000842,0.006268,0.012238,-0.005458,0.025651,-0.001384,-0.027094
age,-0.003965,1.000000,-0.009997,0.028308,-0.030680,-0.011721,0.085472,-0.007201,0.285323
tenure,0.000842,-0.009997,1.000000,-0.012254,0.013444,0.022583,-0.028362,0.007784,-0.014001
balance,0.006268,0.028308,-0.012254,1.000000,-0.304180,-0.014858,-0.010084,0.012797,0.118533
products_number,0.012238,-0.030680,0.013444,-0.304180,1.000000,0.003183,0.009612,0.014204,-0.047820
credit_card,-0.005458,-0.011721,0.022583,-0.014858,0.003183,1.000000,-0.011866,-0.009933,-0.007138
active_member,0.025651,0.085472,-0.028362,-0.010084,0.009612,-0.011866,1.000000,-0.011421,-0.156128
estimated_salary,-0.001384,-0.007201,0.007784,0.012797,0.014204,-0.009933,-0.011421,1.000000,0.012097
churn,-0.027094,0.285323,-0.014001,0.118533,-0.047820,-0.007138,-0.156128,0.012097,1.000000


In [24]:
fig = create_heatmap(data=correlation_matrix,title='Correlation Matrix')
fig.show()

## Correlation Analysis

The correlation matrix shows that age has the strongest positive relationship with churn (0.285), indicating that older customers are more likely to leave the bank.

Active membership has a negative correlation with churn (-0.156), suggesting that engaged customers are less likely to churn.

Balance also shows a weak positive correlation (0.118), while most other variables have little linear relationship with churn.

## Statistical Test: Active Membership and Churn

In this section, we test whether active customers are significantly less likely to churn than inactive customers.

In [25]:
contingency_table = pd.crosstab(df['active_member'], df['churn'])
contingency_table

churn,0,1
active_member,,
0,3547,1302
1,4416,735


In [26]:
chi2, p_value, dof, expected = chi2_contingency(contingency_table)

print(f"Chi-square statistic: {chi2:.4f}")
print(f"P-value: {p_value:.10f}")

Chi-square statistic: 242.9853
P-value: 0.0000000000


## Statistical Test Results

The chi-square test revealed a highly significant relationship between customer activity and churn (p-value < 0.001).

Inactive customers have a churn rate of 26.9%, compared to 14.3% for active customers.

This indicates that customer engagement is a key factor in retention and should be a strategic focus for the bank.

## Predictive Modeling: Logistic Regression

In this section, we build a logistic regression model to predict customer churn and identify the most influential factors.

### Define Features and Target

We separate the explanatory variables from the target variable.

- `X` contains the customer features
- `y` contains the churn variable

In [27]:
X = df.drop(columns=["churn"])
y = df["churn"]

### Identify Numerical and Categorical Features

Numerical features need to be scaled, while categorical features need to be encoded before training the model.

In [28]:
categorical_features = ["country", "gender"]
numeric_features = [
    "credit_score",
    "age",
    "tenure",
    "balance",
    "products_number",
    "credit_card",
    "active_member",
    "estimated_salary",
]

### Build the Preprocessing Pipeline

We use:
- `StandardScaler` to standardize numerical variables
- `OneHotEncoder` to transform categorical variables into numerical columns

In [29]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(drop="first"), categorical_features)
    ]
)

### Create the Logistic Regression Pipeline

We combine preprocessing and logistic regression into one pipeline.

This makes the workflow cleaner and prevents data leakage.

In [30]:
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000))
    ]
)

### Split the Dataset

We split the dataset into training and testing sets.

The model is trained on the training set and evaluated on unseen data from the test set.

In [31]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

### Train the Model

We train the logistic regression model using the training data.

In [32]:
model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transforme

### Generate Predictions

We generate:
- predicted classes using `predict`
- predicted churn probabilities using `predict_proba`

In [33]:
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

### Evaluate Model Performance

We evaluate the model using:
- Accuracy
- ROC AUC
- Classification report
- Confusion matrix

In [34]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_pred_proba))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.808
ROC AUC: 0.77475626628169

Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.97      0.89      1593
           1       0.59      0.19      0.28       407

    accuracy                           0.81      2000
   macro avg       0.71      0.58      0.59      2000
weighted avg       0.78      0.81      0.77      2000


Confusion Matrix:
[[1540   53]
 [ 331   76]]


## Model Performance Interpretation

The logistic regression model achieved an accuracy of 80.8% and a ROC AUC score of 0.775, indicating good overall predictive performance.

However, the recall for churned customers is only 19%, meaning that many customers at risk are not detected.

This highlights the limitations of using accuracy alone and suggests that further improvements, such as class balancing or threshold tuning, are needed to better identify customers likely to leave.

# Business Recommendations and Conclusion

Based on the exploratory analysis, statistical testing, and predictive modeling, several key insights were identified to help the bank reduce customer churn.

    ## Key Findings

1. The overall churn rate is approximately 20%, meaning that one out of five customers leaves the bank.

2. Customer age has the strongest positive correlation with churn, indicating that older customers are more likely to leave.

3. Active customers are significantly less likely to churn.
   - Inactive customers: 26.9% churn rate
   - Active customers: 14.3% churn rate

4. The chi-square test confirmed that the relationship between customer activity and churn is highly statistically significant.

5. The logistic regression model achieved:
   - Accuracy: 80.8%
   - ROC AUC: 0.775

6. The model showed limited recall for churned customers (19%), indicating that further improvements are needed to better detect at-risk customers.

## Business Recommendations

Based on the analysis, the following actions are recommended:

### 1. Increase Customer Engagement
Inactive customers are almost twice as likely to churn. The bank should implement targeted engagement campaigns and personalized offers.

### 2. Monitor High-Risk Customers
Older customers and customers with specific behavioral patterns should be closely monitored.

### 3. Develop Early Warning Systems
The predictive model can be integrated into the bank's systems to identify customers with a high probability of churn.

### 4. Improve Predictive Performance
Future work should focus on handling class imbalance and testing advanced models such as Random Forest or XGBoost.

## Conclusion

This project combined data cleaning, exploratory analysis, statistical testing, and predictive modeling to better understand customer churn in the banking sector.

The results highlight customer engagement as one of the most important factors influencing retention.

By leveraging data-driven insights and predictive analytics, banks can proactively identify at-risk customers and implement effective retention strategies.